# 08 · PyTorch: o treino completo

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Na aula 07 você escreveu o laço de treino. Agora vamos cercá-lo de tudo o que um treino de verdade precisa: dados carregados em lotes, GPU quando houver, validação a cada época, parada na hora certa, modelo salvo, experimentos registrados. O problema muda de natureza: saímos das tabelas e entramos nas **imagens**, onde as redes neurais não têm concorrente.

## O que você será capaz de fazer ao final

1. Escrever um `Dataset` e carregar dados com `DataLoader`.
2. Escrever código que roda em CPU ou GPU sem alterações.
3. Construir uma rede convolucional e acompanhar as formas dos tensores camada a camada.
4. Organizar o treino em funções, com validação, parada antecipada e *checkpoints*.
5. Usar *dropout*, *weight decay*, aumento de dados e agendamento da taxa de aprendizado.
6. Registrar e comparar experimentos, e tornar um treino reprodutível.
7. Avaliar o modelo final no teste e analisar os erros com a matriz de confusão.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import gzip
import random
import time
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEMENTE = 42
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "· dispositivo:", dispositivo)

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11})


def semear(semente=SEMENTE):
    """Fixa as sementes de todos os geradores de números aleatórios usados no treino."""
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)

> 📌 **GPU no Colab:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. O notebook roda em CPU em poucos minutos; com GPU, as redes convolucionais ficam várias vezes mais rápidas.

## 1. Os dados: Fashion-MNIST

O Fashion-MNIST reúne 70 mil fotos de peças de roupa do catálogo da Zalando, em tons de cinza, com 28 × 28 pixels, divididas em 10 classes. É pequeno o bastante para treinar em minutos e difícil o bastante para que as escolhas de modelagem façam diferença.

Vamos baixar os quatro arquivos originais e lê-los nós mesmos, para ver o que há dentro.

In [ ]:
BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/"
PASTA = Path("fashion_mnist")
PASTA.mkdir(exist_ok=True)
for arquivo in ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz", "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]:
    if not (PASTA / arquivo).exists():
        urlretrieve(BASE + arquivo, PASTA / arquivo)


def ler_imagens(caminho):
    with gzip.open(caminho) as f:                       # 16 bytes de cabeçalho; depois, um byte por pixel
        return np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 28, 28)


def ler_rotulos(caminho):
    with gzip.open(caminho) as f:                       # 8 bytes de cabeçalho; depois, um byte por rótulo
        return np.frombuffer(f.read(), dtype=np.uint8, offset=8)


imagens = ler_imagens(PASTA / "train-images-idx3-ubyte.gz")
rotulos = ler_rotulos(PASTA / "train-labels-idx1-ubyte.gz")
imagens_teste = ler_imagens(PASTA / "t10k-images-idx3-ubyte.gz")
rotulos_teste = ler_rotulos(PASTA / "t10k-labels-idx1-ubyte.gz")

NOMES = ["Camiseta", "Calça", "Suéter", "Vestido", "Casaco", "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"]

print("treino:", imagens.shape, imagens.dtype, "· pixels de", imagens.min(), "a", imagens.max())
print("teste: ", imagens_teste.shape, "· exemplos por classe no treino:", np.bincount(rotulos))

In [ ]:
fig, eixos = plt.subplots(2, 10, figsize=(12, 2.9))
for classe in range(10):
    for linha, posicao in enumerate(np.flatnonzero(rotulos == classe)[:2]):
        eixos[linha, classe].imshow(imagens[posicao], cmap="gray_r")
        eixos[linha, classe].axis("off")
    eixos[0, classe].set_title(NOMES[classe], fontsize=9, loc="center", fontweight="normal")
plt.tight_layout()

Uma imagem em tons de cinza é uma matriz de números entre 0 (fundo) e 255 (traço mais forte). As classes são balanceadas: 6 mil exemplos de cada no treino.

> 📌 O pacote `torchvision` entrega este e outros conjuntos em uma linha: `torchvision.datasets.FashionMNIST("dados", download=True)`. Em projetos, use-o. Aqui lemos os arquivos à mão porque o objetivo é entender o que um `Dataset` faz por dentro.

### Ajuste, validação e teste

O conjunto de teste oficial (10 mil imagens) é o nosso cofre. Das 60 mil de treino, separamos 10 mil para validação.

In [ ]:
ordem = torch.randperm(60_000, generator=torch.Generator().manual_seed(SEMENTE)).numpy()     # embaralhamento reprodutível
ind_ajuste, ind_valid = ordem[:50_000], ordem[50_000:]

MEDIA = float(imagens[ind_ajuste].mean() / 255)        # estatísticas calculadas SÓ no ajuste
DESVIO = float(imagens[ind_ajuste].std() / 255)
print(f"ajuste: {len(ind_ajuste)} · validação: {len(ind_valid)} · teste: {len(imagens_teste)} · média {MEDIA:.3f} · desvio {DESVIO:.3f}")

## 2. `Dataset` e `DataLoader`

Na aula 07 fatiamos tensores à mão para formar os mini-lotes. Funciona enquanto os dados cabem na memória e não precisam de preparo. O PyTorch separa o problema em duas peças:

| Peça | Responsabilidade | Você escreve |
|---|---|---|
| **`Dataset`** | saber **quantos** exemplos existem e **entregar um** exemplo, dado o índice | `__len__` e `__getitem__` |
| **`DataLoader`** | montar os **lotes**: embaralhar, agrupar, carregar em paralelo | nada: só configura |

Essa separação é o que permite treinar com um milhão de imagens em disco: o `Dataset` abre a imagem pedida, e só ela.

In [ ]:
class ImagensDeRoupas(Dataset):
    """Devolve pares (imagem, rótulo) prontos para a rede."""

    def __init__(self, imagens, rotulos, aumentar=False):
        x = torch.from_numpy(imagens.copy()).float().div(255)        # uint8 em 0..255 -> float32 em 0..1
        self.x = ((x - MEDIA) / DESVIO).unsqueeze(1)                  # padroniza; (N, 28, 28) -> (N, 1, 28, 28)
        self.y = torch.from_numpy(rotulos.copy()).long()              # rótulos de classe: int64
        self.aumentar = aumentar                                      # usado na seção 7

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        imagem = self.x[i]
        if self.aumentar and torch.rand(()) < 0.5:
            imagem = imagem.flip(-1)                                  # espelha na horizontal, metade das vezes
        return imagem, self.y[i]


dados_ajuste = ImagensDeRoupas(imagens[ind_ajuste], rotulos[ind_ajuste])
dados_valid = ImagensDeRoupas(imagens[ind_valid], rotulos[ind_valid])
dados_teste = ImagensDeRoupas(imagens_teste, rotulos_teste)

imagem, rotulo = dados_ajuste[0]                                      # colchetes chamam __getitem__
print("len:", len(dados_ajuste), "· imagem:", tuple(imagem.shape), imagem.dtype, "· rótulo:", rotulo.item(), f"({NOMES[rotulo]})")

A forma `(1, 28, 28)` segue a convenção do PyTorch para imagens: **canais, altura, largura**. Uma imagem colorida teria 3 canais.

In [ ]:
carregador_ajuste = DataLoader(dados_ajuste, batch_size=128, shuffle=True)      # embaralha a cada época
carregador_valid = DataLoader(dados_valid, batch_size=1000)                     # avaliação: sem embaralhar, lotes maiores
carregador_teste = DataLoader(dados_teste, batch_size=1000)

lote_x, lote_y = next(iter(carregador_ajuste))                                  # um lote
print("lote de imagens:", tuple(lote_x.shape), "· lote de rótulos:", tuple(lote_y.shape))
print("lotes por época:", len(carregador_ajuste), "=", len(dados_ajuste), "/ 128, arredondado para cima")

Um lote de imagens tem quatro dimensões: **(lote, canais, altura, largura)**.

| Argumento do `DataLoader` | Para quê |
|---|---|
| `batch_size` | exemplos por lote |
| `shuffle=True` | embaralhar a cada época; **só no ajuste** |
| `num_workers` | processos paralelos para carregar os dados |
| `pin_memory=True` | acelera a cópia para a GPU |
| `drop_last=True` | descarta o último lote, se incompleto |

### 🔮 Preveja: mais processos, mais rápido?

Percorrer o conjunto de ajuste com `num_workers=2` é mais rápido que com `num_workers=0`?

In [ ]:
for trabalhadores in [0, 2]:
    carregador = DataLoader(dados_ajuste, batch_size=128, shuffle=True, num_workers=trabalhadores)
    inicio = time.perf_counter()
    for _ in carregador:
        pass
    print(f"num_workers={trabalhadores}: {time.perf_counter() - inicio:.2f} s para percorrer uma época")

Aqui, não. Nossos dados já estão prontos na memória e o `__getitem__` só devolve uma fatia; criar processos e transferir os lotes entre eles custa mais do que o trabalho poupado. Os `workers` compensam quando cada exemplo dá trabalho: abrir um JPEG do disco, redimensionar, aplicar transformações. **Meça antes de configurar.**

### ✏️ Exercício 1 (★★): um `Dataset` só de calçados

Escreva a classe `SoCalcados`, que recebe os arrays `imagens` e `rotulos` e fica apenas com as classes 5 (Sandália), 7 (Tênis) e 9 (Bota).

- `__len__` devolve o número de exemplos dessas três classes.
- `__getitem__` devolve a imagem como tensor `float32` de forma `(1, 28, 28)`, com valores entre 0 e 1, e o rótulo como tensor `int64` **renumerado**: 5 → 0, 7 → 1, 9 → 2.

In [ ]:
# ✏️ Exercício 1
class SoCalcados(Dataset):
    # SEU CÓDIGO AQUI
    def __init__(self, imagens, rotulos):
        ...
    
    def __len__(self):
        ...
    
    def __getitem__(self, i):
        ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
_calcados = SoCalcados(imagens_teste, rotulos_teste)
assert len(_calcados) == int(np.isin(rotulos_teste, [5, 7, 9]).sum()) == 3000, "o conjunto de teste tem 3000 calçados"
_x, _y = _calcados[0]
assert tuple(_x.shape) == (1, 28, 28) and _x.dtype == torch.float32 and 0 <= _x.min() and _x.max() <= 1, "confira forma, tipo e escala da imagem"
assert _y.dtype == torch.int64 and {int(_calcados[i][1]) for i in range(300)} == {0, 1, 2}, "os rótulos devem ser 0, 1 e 2, em int64"
_lx, _ly = next(iter(DataLoader(_calcados, batch_size=32)))
assert tuple(_lx.shape) == (32, 1, 28, 28) and tuple(_ly.shape) == (32,)
_primeiro = int(np.flatnonzero(np.isin(rotulos_teste, [5, 7, 9]))[0])
assert int(_y) == {5: 0, 7: 1, 9: 2}[int(rotulos_teste[_primeiro])], "a renumeração não corresponde aos rótulos originais"
print("Exercício 1 correto: 3000 calçados, prontos para um DataLoader.")

## 3. Dispositivo: o mesmo código em CPU e GPU

O padrão tem três partes, e vale para qualquer projeto:

```python
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"     # 1. descobrir, uma vez
modelo = modelo.to(dispositivo)                                   # 2. mover o modelo, uma vez
for xb, yb in carregador:
    xb, yb = xb.to(dispositivo), yb.to(dispositivo)               # 3. mover cada lote
```

Os dados ficam na memória comum (ou no disco) e só o lote da vez vai para a GPU. É isso que permite treinar com conjuntos maiores que a memória da placa.

## 4. O treino em funções

Dividimos o laço da aula 07 em duas funções: uma que treina por uma época e outra que avalia.

A perda agora é a `nn.CrossEntropyLoss`: a rede devolve **10 *logits*** por imagem, e o rótulo é o índice da classe. A classe prevista é a do maior *logit* (`argmax`).

In [ ]:
def treinar_uma_epoca(modelo, carregador, funcao_perda, otimizador):
    """Uma passagem pelos dados de ajuste. Devolve a perda média e a acurácia."""
    modelo.train()
    soma_perda, acertos, total = 0.0, 0, 0
    for xb, yb in carregador:
        xb, yb = xb.to(dispositivo), yb.to(dispositivo)

        otimizador.zero_grad()                        # 1. zerar
        logits = modelo(xb)                           # 2. prever: (lote, 10)
        perda = funcao_perda(logits, yb)              # 3. medir a perda
        perda.backward()                              # 4. retropropagar
        otimizador.step()                             # 5. atualizar

        soma_perda += perda.item() * len(yb)
        acertos += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    return soma_perda / total, acertos / total


@torch.no_grad()                                      # o decorador desliga os gradientes na função inteira
def avaliar(modelo, carregador, funcao_perda):
    """Perda média e acurácia em um conjunto, sem alterar o modelo."""
    modelo.eval()
    soma_perda, acertos, total = 0.0, 0, 0
    for xb, yb in carregador:
        xb, yb = xb.to(dispositivo), yb.to(dispositivo)
        logits = modelo(xb)
        soma_perda += funcao_perda(logits, yb).item() * len(yb)
        acertos += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    return soma_perda / total, acertos / total


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

### Uma referência: MLP

Antes de qualquer arquitetura especial, um modelo simples: achatar a imagem em um vetor de 784 números e passar por uma camada oculta.

In [ ]:
semear()
mlp = nn.Sequential(
    nn.Flatten(),                  # (lote, 1, 28, 28) -> (lote, 784)
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),            # 10 logits, um por classe
).to(dispositivo)

funcao_perda = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(mlp.parameters(), lr=1e-3)

print(f"{contar_parametros(mlp)} parâmetros · antes de treinar: acurácia de validação {avaliar(mlp, carregador_valid, funcao_perda)[1]:.1%} (sorteio: 10%)")
for epoca in range(1, 4):
    perda_ajuste, acuracia_ajuste = treinar_uma_epoca(mlp, carregador_ajuste, funcao_perda, otimizador)
    perda_valid, acuracia_valid = avaliar(mlp, carregador_valid, funcao_perda)
    print(f"época {epoca} · ajuste: perda {perda_ajuste:.4f}, acurácia {acuracia_ajuste:.1%} · validação: perda {perda_valid:.4f}, acurácia {acuracia_valid:.1%}")

Três épocas, poucos segundos, e a rede acerta mais de 85% das imagens de validação. É o número a bater.

### ✏️ Exercício 2 (★★): acurácia por classe

A acurácia geral esconde diferenças entre as classes. Escreva `acuracia_por_classe(modelo, carregador)`, que devolve um tensor com 10 números: para cada classe, a proporção de imagens **daquela classe** classificadas corretamente. Lembre-se do modo de avaliação, de desligar os gradientes e de mover os lotes para o dispositivo.

In [ ]:
# ✏️ Exercício 2
@torch.no_grad()
def acuracia_por_classe(modelo, carregador):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
_por_classe = acuracia_por_classe(mlp, carregador_valid)
assert tuple(_por_classe.shape) == (10,) and (_por_classe >= 0).all() and (_por_classe <= 1).all()
_contagem = torch.bincount(dados_valid.y, minlength=10).float()
_geral = avaliar(mlp, carregador_valid, funcao_perda)[1]
assert abs((_por_classe.cpu() * _contagem).sum().item() / _contagem.sum().item() - _geral) < 1e-4, "a média ponderada das acurácias por classe deve ser igual à acurácia geral"
assert not mlp.training, "a função deve colocar o modelo em modo de avaliação"
print("Exercício 2 correto. Acurácia do MLP por classe, na validação:")
pd.Series(_por_classe.cpu().numpy(), index=NOMES).sort_values().round(3).to_frame("acurácia").T

## 5. Redes convolucionais

O MLP começa achatando a imagem, e com isso **joga fora a informação de vizinhança**: para ele, o pixel (3, 4) e o pixel (3, 5) são duas entradas quaisquer. Embaralhar os pixels de todas as imagens da mesma maneira não mudaria nada no resultado.

Uma **camada convolucional** parte de duas ideias:

- **Localidade:** padrões visuais (bordas, cantos, texturas) são formados por pixels vizinhos. Um **filtro** pequeno, de 3 × 3, olha uma vizinhança por vez.
- **Compartilhamento de pesos:** o mesmo filtro percorre a imagem inteira. Uma borda é uma borda no canto ou no centro.

O resultado de passar um filtro pela imagem é um **mapa de ativação**: outra imagem, que acende onde o padrão aparece. Vejamos com um filtro feito à mão.

In [ ]:
exemplo = dados_ajuste.x[np.flatnonzero(rotulos[ind_ajuste] == 1)[0]].unsqueeze(0)       # uma calça, com forma (1, 1, 28, 28)

filtro_vertical = torch.tensor([[-1.0, 0.0, 1.0],
                                [-1.0, 0.0, 1.0],
                                [-1.0, 0.0, 1.0]]).reshape(1, 1, 3, 3)                   # responde a mudanças da esquerda para a direita
filtro_horizontal = filtro_vertical.transpose(2, 3)

fig, eixos = plt.subplots(1, 3, figsize=(8, 2.9))
eixos[0].imshow(exemplo[0, 0], cmap="gray_r"); eixos[0].set_title("Imagem")
eixos[1].imshow(nn.functional.conv2d(exemplo, filtro_vertical)[0, 0].abs(), cmap="gray_r"); eixos[1].set_title("Filtro de bordas verticais")
eixos[2].imshow(nn.functional.conv2d(exemplo, filtro_horizontal)[0, 0].abs(), cmap="gray_r"); eixos[2].set_title("Filtro de bordas horizontais")
for ax in eixos:
    ax.axis("off")
plt.tight_layout()

Em uma rede convolucional, **os filtros são os parâmetros**: a rede aprende quais padrões vale a pena detectar. As primeiras camadas aprendem bordas e texturas; as seguintes combinam esses padrões em partes maiores.

| Camada | O que faz | Efeito na forma |
|---|---|---|
| `nn.Conv2d(entrada, saída, kernel_size=3, padding=1)` | aplica `saída` filtros 3 × 3 | muda o número de canais; com `padding=1`, mantém altura e largura |
| `nn.MaxPool2d(2)` | fica com o maior valor de cada bloco 2 × 2 | divide altura e largura por 2 |
| `nn.Flatten()` | achata tudo, menos o eixo do lote | `(N, C, A, L)` → `(N, C·A·L)` |

Acompanhar as formas é a habilidade prática mais importante desta seção. Vamos passar um lote camada a camada.

In [ ]:
x = lote_x[:4]
print(f"{'entrada':<12}{tuple(x.shape)}")
for camada in [nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
               nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
               nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10)]:
    x = camada(x)
    print(f"{camada.__class__.__name__:<12}{tuple(x.shape)}")

A imagem entra com 1 canal e 28 × 28 pixels e chega ao `Flatten` com 32 canais de 7 × 7: menos resolução, mais "tipos de padrão". O número `32 * 7 * 7 = 1568` na primeira camada linear vem dessa conta; errar nele é o erro de forma mais comum em redes convolucionais.

In [ ]:
class RedeConvolucional(nn.Module):
    def __init__(self, p_dropout=0.0):
        super().__init__()
        self.extrator = nn.Sequential(                             # das imagens aos mapas de ativação
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classificador = nn.Sequential(                        # dos mapas de ativação às classes
            nn.Flatten(),
            nn.Dropout(p_dropout),                                 # com p = 0 não faz nada; usado na seção 7
            nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classificador(self.extrator(x))


print(f"MLP: {contar_parametros(mlp)} parâmetros · rede convolucional: {contar_parametros(RedeConvolucional())} parâmetros")

As duas redes têm praticamente o mesmo número de parâmetros. A diferença de desempenho que veremos vem da **estrutura**, e não do tamanho.

### ✏️ Exercício 3 (★★): a conta das formas

Complete a classe `RedePequena` com a seguinte sequência: convolução de 1 para 8 canais (3 × 3, `padding=1`), ReLU, *max pooling* 2 × 2, convolução de 8 para 16 canais (3 × 3, `padding=1`), ReLU, *max pooling* 2 × 2, `Flatten` e uma camada linear para 10 saídas. O número de entradas da camada linear é por sua conta.

In [ ]:
# ✏️ Exercício 3
class RedePequena(nn.Module):
    def __init__(self):
        super().__init__()
        # SEU CÓDIGO AQUI
        self.camadas = ...

    def forward(self, x):
        return self.camadas(x)

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_rede = RedePequena()
assert tuple(_rede(lote_x[:5]).shape) == (5, 10), "a saída deve ter 10 logits por imagem"
assert contar_parametros(_rede) == (8 * 9 + 8) + (16 * 8 * 9 + 16) + (16 * 7 * 7 * 10 + 10), "o número de parâmetros não corresponde à arquitetura pedida"
print(f"Exercício 3 correto: {contar_parametros(_rede)} parâmetros, dos quais {16 * 7 * 7 * 10 + 10} na camada linear.")

## 6. O treino completo

A função `ajustar`, abaixo, é o laço de épocas com tudo o que faltava:

| Recurso | O que faz | Por quê |
|---|---|---|
| **Validação a cada época** | mede perda e acurácia fora do ajuste | é a única forma de ver o sobreajuste |
| ***Checkpoint*** | grava em disco os parâmetros sempre que a validação melhora | o modelo final é o **melhor**, e não o último; e uma queda de energia não custa o treino |
| **Parada antecipada** | interrompe depois de `paciencia` épocas sem melhora | economiza tempo e evita sobreajuste |
| **Registro** | guarda o histórico e o resultado em uma tabela de experimentos | sem registro, não há como comparar |

In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter       # já vem instalado no Colab
except ImportError:
    SummaryWriter = None

PASTA_MODELOS = Path("modelos")
PASTA_MODELOS.mkdir(exist_ok=True)
experimentos = {}                                            # nome do experimento -> resumo
modelos = {}                                                 # nome do experimento -> modelo treinado


def ajustar(modelo, nome, epocas=8, taxa=1e-3, weight_decay=0.0, paciencia=3, carregador=None, cosseno=False):
    """Treina `modelo`, salva o melhor checkpoint em modelos/<nome>.pt e registra o experimento."""
    carregador = carregador or carregador_ajuste
    modelo.to(dispositivo)
    funcao_perda = nn.CrossEntropyLoss()
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=taxa, weight_decay=weight_decay)
    agendador = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=epocas) if cosseno else None
    escritor = SummaryWriter(f"runs/{nome}") if SummaryWriter else None
    caminho = PASTA_MODELOS / f"{nome}.pt"

    historico, melhor_perda, sem_melhora = [], float("inf"), 0
    inicio = time.perf_counter()
    for epoca in range(1, epocas + 1):
        perda_ajuste, acuracia_ajuste = treinar_uma_epoca(modelo, carregador, funcao_perda, otimizador)
        perda_valid, acuracia_valid = avaliar(modelo, carregador_valid, funcao_perda)
        if agendador:
            agendador.step()                                 # ajusta a taxa de aprendizado para a próxima época
        historico.append({"epoca": epoca, "perda_ajuste": perda_ajuste, "perda_valid": perda_valid,
                          "acuracia_ajuste": acuracia_ajuste, "acuracia_valid": acuracia_valid})
        if escritor:
            escritor.add_scalar("perda/ajuste", perda_ajuste, epoca)
            escritor.add_scalar("perda/validacao", perda_valid, epoca)
            escritor.add_scalar("acuracia/validacao", acuracia_valid, epoca)

        melhorou = perda_valid < melhor_perda
        if melhorou:                                         # checkpoint: tudo o que é preciso para usar ou retomar
            melhor_perda, sem_melhora = perda_valid, 0
            torch.save({"modelo": modelo.state_dict(), "otimizador": otimizador.state_dict(), "epoca": epoca,
                        "perda_valid": perda_valid, "acuracia_valid": acuracia_valid}, caminho)
        else:
            sem_melhora += 1
        print(f"[{nome}] época {epoca:>2} · ajuste {perda_ajuste:.4f} ({acuracia_ajuste:.1%}) · validação {perda_valid:.4f} ({acuracia_valid:.1%})"
              + ("  ← melhor até aqui, salvo" if melhorou else ""))
        if sem_melhora >= paciencia:                         # parada antecipada
            print(f"[{nome}] parada antecipada: {paciencia} épocas sem melhora na validação.")
            break

    ponto = torch.load(caminho, map_location=dispositivo, weights_only=True)
    modelo.load_state_dict(ponto["modelo"])                  # o modelo volta ao estado da melhor época
    if escritor:
        escritor.close()
    modelos[nome] = modelo
    experimentos[nome] = {"parâmetros": contar_parametros(modelo), "taxa": taxa, "weight_decay": weight_decay,
                          "épocas": epoca, "melhor época": ponto["epoca"], "perda valid": ponto["perda_valid"],
                          "acurácia valid": ponto["acuracia_valid"], "segundos": round(time.perf_counter() - inicio, 1)}
    return pd.DataFrame(historico)

Primeiro o MLP, com um limite generoso de épocas, para ver a parada antecipada agir.

In [ ]:
semear()
historico_mlp = ajustar(nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)), "mlp", epocas=25)

⏳ Agora a rede convolucional. Em CPU, cada época leva alguns segundos; a célula toda, cerca de um minuto.

In [ ]:
semear()
historico_cnn = ajustar(RedeConvolucional(), "cnn", epocas=8)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11, 3.4))
for historico, nome, cor in [(historico_mlp, "MLP", CINZA), (historico_cnn, "rede convolucional", AZUL)]:
    eixos[0].plot(historico["epoca"], historico["perda_ajuste"], color=cor, ls="--", lw=1)
    eixos[0].plot(historico["epoca"], historico["perda_valid"], color=cor, label=nome)
    eixos[1].plot(historico["epoca"], historico["acuracia_valid"], color=cor, label=nome)
eixos[0].set_xlabel("época"); eixos[0].set_ylabel("perda"); eixos[0].legend(frameon=False)
eixos[0].set_title("Perda: validação (linha cheia) e ajuste (tracejada)")
eixos[1].set_xlabel("época"); eixos[1].set_ylabel("acurácia de validação"); eixos[1].legend(frameon=False)
eixos[1].set_title("A rede convolucional aprende mais, com os mesmos parâmetros")
plt.tight_layout()
pd.DataFrame(experimentos).T

**Leituras.**

- No MLP, a perda de ajuste (tracejada) continua caindo enquanto a de validação estaciona e oscila. A parada antecipada interrompeu o treino na época 12, e o modelo que ficou é o da época 9, lido do *checkpoint*.
- A rede convolucional termina cerca de dois pontos percentuais acima, com o mesmo número de parâmetros, e ainda estava melhorando quando o limite de épocas chegou.
- A tabela de experimentos, ao final, registra o que foi feito. Ela vai crescer.

### O que há em um *checkpoint*, e como retomar um treino

In [ ]:
ponto = torch.load(PASTA_MODELOS / "cnn.pt", map_location=dispositivo, weights_only=True)
print("chaves:", list(ponto), "· época:", ponto["epoca"], f"· acurácia de validação: {ponto['acuracia_valid']:.4f}")
print("arquivo:", f"{(PASTA_MODELOS / 'cnn.pt').stat().st_size / 1e6:.1f} MB")

# Para USAR o modelo, bastam a arquitetura e os parâmetros
restaurada = RedeConvolucional().to(dispositivo)
restaurada.load_state_dict(ponto["modelo"])
print(f"acurácia da rede restaurada: {avaliar(restaurada, carregador_valid, nn.CrossEntropyLoss())[1]:.4f}")

# Para RETOMAR o treino, o otimizador também precisa voltar ao estado em que estava
otimizador_restaurado = torch.optim.AdamW(restaurada.parameters(), lr=1e-3)
otimizador_restaurado.load_state_dict(ponto["otimizador"])
perda_ajuste, acuracia_ajuste = treinar_uma_epoca(restaurada, carregador_ajuste, nn.CrossEntropyLoss(), otimizador_restaurado)
print(f"uma época a mais, a partir da época {ponto['epoca']}: perda de ajuste {perda_ajuste:.4f}")

O Adam guarda médias móveis dos gradientes de cada parâmetro. Retomar o treino com um otimizador novo descarta essas médias e costuma causar um degrau na curva de perda; por isso o `state_dict` do otimizador vai junto no *checkpoint*.

> ⚠️ **No Colab, os arquivos somem quando a sessão termina.** Para treinos longos, grave os *checkpoints* no Google Drive (`from google.colab import drive; drive.mount("/content/drive")`) ou em outro armazenamento persistente.

## 7. Regularização e agendamento

A distância entre as curvas de ajuste e de validação diz que a rede está decorando. Quatro ferramentas atacam esse problema, e todas já estão ligadas ao nosso código:

| Técnica | Ideia | Onde está no código |
|---|---|---|
| ***Dropout*** | durante o treino, zera ao acaso uma fração das ativações; a rede não pode depender de um neurônio só | `RedeConvolucional(p_dropout=0.3)` |
| ***Weight decay*** | puxa os pesos em direção a zero a cada passo | `ajustar(..., weight_decay=1e-4)`, com o otimizador AdamW |
| **Aumento de dados** | apresenta variações plausíveis das imagens (aqui, espelhadas) | `ImagensDeRoupas(..., aumentar=True)` |
| **Agendamento da taxa** | começa com passos grandes e termina com passos pequenos | `ajustar(..., cosseno=True)` |

### *Dropout* e os dois modos do modelo

O *dropout* é o motivo de existirem `modelo.train()` e `modelo.eval()`: ele só pode agir durante o treino.

In [ ]:
torch.manual_seed(0)
dropout = nn.Dropout(p=0.5)
uns = torch.ones(10)

dropout.train()
print("modo de treino:   ", dropout(uns))       # metade vira zero; o restante é multiplicado por 2, para manter a média
dropout.eval()
print("modo de avaliação:", dropout(uns))       # não faz nada

Esquecer o `eval()` na avaliação deixa o *dropout* ligado: as previsões mudam a cada chamada e o desempenho medido piora. Esquecer o `train()` ao voltar ao treino desliga a regularização.

### Um experimento com as quatro técnicas

⏳ Mais um treino de cerca de um minuto em CPU.

In [ ]:
semear()
carregador_aumentado = DataLoader(ImagensDeRoupas(imagens[ind_ajuste], rotulos[ind_ajuste], aumentar=True), batch_size=128, shuffle=True)

historico_reg = ajustar(RedeConvolucional(p_dropout=0.3), "cnn_regularizada", epocas=8,
                        weight_decay=1e-4, carregador=carregador_aumentado, cosseno=True)

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.4))
for historico, nome, cor in [(historico_cnn, "sem regularização", AZUL), (historico_reg, "com regularização", VERDE)]:
    ax.plot(historico["epoca"], historico["perda_ajuste"], color=cor, ls="--", lw=1)
    ax.plot(historico["epoca"], historico["perda_valid"], color=cor, label=nome)
ax.set_xlabel("época"); ax.set_ylabel("perda"); ax.legend(frameon=False)
ax.set_title("Validação (cheia) e ajuste (tracejada), com e sem regularização")
print(pd.DataFrame(experimentos).T[["melhor época", "perda valid", "acurácia valid", "segundos"]])

Duas observações.

- Na rede regularizada, a perda de ajuste (tracejada) fica **acima** da de validação durante quase todo o treino, e as duas se encontram no fim. O treino ficou mais difícil (ativações zeradas, imagens espelhadas), enquanto a avaliação é feita com a rede inteira e as imagens originais. A distância entre as curvas, que indicava sobreajuste, desapareceu.
- Em oito épocas, porém, a rede regularizada termina um pouco **atrás** na validação (90,0% de acurácia contra 90,9%). Não há contradição: regularizar torna o aprendizado mais lento. O retorno aparece em treinos longos e redes maiores, quando o modelo sem regularização já parou de melhorar e começou a decorar.

Na preparação desta aula, as duas redes foram treinadas por 30 épocas. A rede sem regularização teve a menor perda de validação na época 13 (91,7% de acurácia) e daí em diante a perda só subiu. A regularizada continuou melhorando até o fim e chegou a 92,6%.

A lição prática: regularização é remédio para sobreajuste, e seu efeito aparece quando o treino é longo o bastante para o sobreajuste se instalar.

## 8. Registro de experimentos e reprodutibilidade

Depois de dez experimentos, ninguém lembra qual taxa de aprendizado deu qual resultado. Três níveis de registro, do mais simples ao mais completo:

**1. Uma tabela.** É o que o dicionário `experimentos` faz. Salve-a junto com o código.

In [ ]:
tabela = pd.DataFrame(experimentos).T
tabela.to_csv("experimentos.csv")
tabela.sort_values("acurácia valid", ascending=False)

**2. TensorBoard.** A função `ajustar` já gravou as curvas de cada experimento na pasta `runs/`. Para abrir o painel no Colab, execute em uma célula:

```python
%load_ext tensorboard
%tensorboard --logdir runs
```

O painel mostra as curvas de todos os experimentos sobrepostas, e se atualiza durante o treino.

**3. Um serviço de acompanhamento**, como Weights & Biases ou MLflow. Guardam curvas, hiperparâmetros, versão do código e arquivos do modelo, e permitem comparar experimentos de uma equipe inteira. O esqueleto com o W&B é este (exige conta gratuita e uma chave, guardada nos *Secrets* do Colab):

```python
# %pip install -q wandb
import wandb
from google.colab import userdata

wandb.login(key=userdata.get("WANDB_API_KEY"))                    # a chave nunca aparece no notebook
execucao = wandb.init(project="pam-fashion-mnist", name="cnn", config={"taxa": 1e-3, "lote": 128, "epocas": 8})
# dentro do laço de épocas:
#     wandb.log({"perda/ajuste": perda_ajuste, "perda/validacao": perda_valid, "acuracia/validacao": acuracia_valid}, step=epoca)
execucao.finish()
```

### Reprodutibilidade

Para que outra pessoa (ou você, daqui a seis meses) chegue ao mesmo resultado:

| Item | Como |
|---|---|
| Sementes | `semear()` antes de criar o modelo e antes de treinar |
| Divisão dos dados | índices gerados com semente fixa, como em `ind_ajuste` e `ind_valid` |
| Versões | registrar `torch.__version__` e as demais; arquivo de dependências no repositório |
| Hiperparâmetros | na tabela de experimentos, e não na memória |
| Código | um *commit* por experimento relevante |

> ⚠️ **Limite:** mesmo com tudo isso, resultados em GPU podem variar na última casa decimal de uma execução para outra, porque algumas operações não são determinísticas. Resultados entre CPU e GPU, ou entre versões do PyTorch, podem diferir um pouco mais. Reprodutibilidade em redes neurais é "mesma conclusão", raramente "mesmos dígitos".

### ✏️ Exercício 4 (★★★): o seu experimento

Rode um experimento novo, com o nome `"ex4"`, tentando superar a acurácia de validação dos anteriores com **no máximo 8 épocas**. Guarde o histórico em `historico_ex4`.

Ideias: outra taxa de aprendizado; outro valor de *dropout*; mais filtros nas convoluções; uma camada `nn.BatchNorm2d` depois de cada convolução; outro tamanho de lote. Mude **uma coisa por vez** em relação a um experimento existente, para saber a que atribuir o resultado.

In [ ]:
# ✏️ Exercício 4
semear()
# SEU CÓDIGO AQUI
historico_ex4 = ajustar(..., "ex4", epocas=8)

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
assert "ex4" in experimentos and len(historico_ex4) <= 8, "registre o experimento com o nome 'ex4' e no máximo 8 épocas"
assert experimentos["ex4"]["acurácia valid"] > 0.85, "a acurácia ficou abaixo da do MLP: revise a configuração"
_melhor_anterior = max(v["acurácia valid"] for k, v in experimentos.items() if k != "ex4")
print(f"Exercício 4 registrado: acurácia de validação {experimentos['ex4']['acurácia valid']:.4f} (melhor dos anteriores: {_melhor_anterior:.4f}).")
pd.DataFrame(experimentos).T[["parâmetros", "melhor época", "perda valid", "acurácia valid", "segundos"]]

**Pergunta escrita.** O que você mudou, em relação a qual experimento, e por quê? O resultado confirmou a sua expectativa? A diferença observada é grande o bastante para você confiar nela?

✏️ **Sua resposta:**

_Escreva aqui._

## 9. Abrindo o cofre: teste e análise de erros

Escolhemos o melhor dos três experimentos guiados **pela validação** e, só agora, medimos no teste.

In [ ]:
GUIADOS = ["mlp", "cnn", "cnn_regularizada"]
melhor_nome = max(GUIADOS, key=lambda nome: experimentos[nome]["acurácia valid"])
melhor_modelo = modelos[melhor_nome]

perda_teste, acuracia_teste = avaliar(melhor_modelo, carregador_teste, nn.CrossEntropyLoss())
print(f"Escolhido pela validação: {melhor_nome} ({experimentos[melhor_nome]['acurácia valid']:.2%}) · acurácia no TESTE: {acuracia_teste:.2%}")

A acurácia resume. Para entender **onde** o modelo erra, usamos a **matriz de confusão**: linhas são as classes verdadeiras, colunas são as previstas.

In [ ]:
from sklearn.metrics import confusion_matrix


@torch.no_grad()
def prever(modelo, carregador):
    modelo.eval()
    return torch.cat([modelo(xb.to(dispositivo)).argmax(dim=1).cpu() for xb, _ in carregador]).numpy()


previstos = prever(melhor_modelo, carregador_teste)
matriz = confusion_matrix(rotulos_teste, previstos)

fig, ax = plt.subplots(figsize=(6.6, 5.4))
ax.imshow(matriz, cmap="Blues")
for i in range(10):
    for j in range(10):
        if matriz[i, j] > 0:
            ax.text(j, i, matriz[i, j], ha="center", va="center", fontsize=8, color="white" if matriz[i, j] > 500 else "black")
ax.set_xticks(range(10), NOMES, rotation=45, ha="right"); ax.set_yticks(range(10), NOMES)
ax.set_xlabel("classe prevista"); ax.set_ylabel("classe verdadeira")
ax.spines[:].set_visible(False)
ax.set_title("Os erros se concentram nas peças de tronco")
plt.tight_layout()

Cada classe tem 1000 imagens no teste; a diagonal mostra os acertos. Calças, bolsas e calçados quase não geram erros. As confusões estão entre camiseta, camisa, suéter, casaco e vestido: peças que, em 28 × 28 pixels e sem cor, se parecem.

In [ ]:
erros = np.flatnonzero(previstos != rotulos_teste)
fig, eixos = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, posicao in zip(eixos.ravel(), erros[:16]):
    ax.imshow(imagens_teste[posicao], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"{NOMES[rotulos_teste[posicao]]}\n→ {NOMES[previstos[posicao]]}", fontsize=8, loc="center", fontweight="normal")
plt.suptitle("Alguns erros: classe verdadeira → classe prevista", x=0.01, ha="left", fontsize=11, fontweight="bold")
plt.tight_layout()

Olhar os erros é parte do trabalho. Vários deles são difíceis também para uma pessoa, e alguns rótulos são discutíveis. Isso indica que a acurácia tem um teto, dado pela qualidade das imagens e dos rótulos, e que insistir em arquiteturas maiores rende cada vez menos.

### ✏️ Exercício 5 (★★): os números da matriz

A partir de `matriz` (10 × 10) e de `NOMES`, obtenha:

- `pior_classe`: o **nome** da classe com a menor proporção de acertos (acertos da classe divididos pelo total de imagens da classe);
- `par_mais_confundido`: uma tupla `(nome da classe verdadeira, nome da classe prevista)` com o erro mais frequente, isto é, a maior contagem fora da diagonal.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
pior_classe = ...
par_mais_confundido = ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert pior_classe in NOMES and len(par_mais_confundido) == 2 and all(nome in NOMES for nome in par_mais_confundido)
_proporcoes = matriz.diagonal() / matriz.sum(axis=1)
assert _proporcoes[NOMES.index(pior_classe)] <= _proporcoes.min() + 1e-12, "há uma classe com proporção de acertos menor"
_v, _p = NOMES.index(par_mais_confundido[0]), NOMES.index(par_mais_confundido[1])
assert _v != _p and matriz[_v, _p] >= (matriz - np.diag(matriz.diagonal())).max(), "há um erro mais frequente que esse"
print(f"Exercício 5 correto: pior classe = {pior_classe} ({_proporcoes[NOMES.index(pior_classe)]:.1%} de acertos) · erro mais comum: {par_mais_confundido[0]} classificada como {par_mais_confundido[1]} ({matriz[_v, _p]} vezes).")

## 10. Para levar

### Erros comuns

| Erro | Consequência | Como evitar |
|---|---|---|
| `shuffle=True` na validação ou no teste | nenhum dano à métrica, mas resultados difíceis de comparar exemplo a exemplo | embaralhar só o ajuste |
| Normalizar com estatísticas do conjunto inteiro | vazamento (pequeno) da validação e do teste | média e desvio calculados no ajuste |
| Esquecer `.to(dispositivo)` no lote | `Expected all tensors to be on the same device` | mover modelo uma vez e cada lote no laço |
| Avaliar sem `modelo.eval()` | *dropout* ligado: métricas piores e instáveis | `eval()` dentro da função de avaliação |
| Avaliar sem `torch.no_grad()` | memória e tempo desperdiçados | decorador `@torch.no_grad()` |
| Ficar com o modelo da última época | modelo sobreajustado | *checkpoint* do melhor e recarga no fim |
| Salvar só os parâmetros do modelo e tentar retomar o treino | degrau na curva de perda | salvar também o otimizador e a época |
| Número de entradas errado depois do `Flatten` | `shapes cannot be multiplied` | passar um lote e imprimir as formas |
| Mudar várias coisas entre dois experimentos | não se sabe o que causou a diferença | uma mudança por vez, tudo registrado |
| Escolher o modelo olhando o teste | estimativa final otimista | escolher pela validação; teste uma vez |

### Resumo

- **`Dataset`** entrega um exemplo; **`DataLoader`** monta os lotes. A separação permite trabalhar com dados maiores que a memória.
- O código fica independente de dispositivo com três linhas: descobrir, mover o modelo, mover cada lote.
- **Convoluções** exploram a vizinhança dos pixels e compartilham pesos; *pooling* reduz a resolução. Acompanhe as formas camada a camada.
- O treino completo tem **validação a cada época**, ***checkpoint* do melhor modelo** e **parada antecipada**.
- ***Dropout*, *weight decay*, aumento de dados e agendamento da taxa** combatem o sobreajuste; `train()` e `eval()` existem por causa deles.
- **Registre** cada experimento (tabela, TensorBoard ou um serviço) e fixe sementes, divisão e versões.
- O teste é aberto uma vez; a **matriz de confusão** e os exemplos errados dizem o que a acurácia esconde.

### O que vem a seguir

Treinar do zero funciona para um problema deste tamanho. Para texto e para imagens reais, o caminho é partir de modelos **pré-treinados**. O notebook 09 apresenta o ecossistema Hugging Face.

### Para casa

A lista de exercícios está em `08_casa.ipynb`.